# Multi-Layer Perceptron
This notebook contains the code for training and evaluating a Multi-Layer Perceptron (MLP) model for binary failure detection and multiclass prediction of the failure mode.

### Imports

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
from src.splits import load_split
from src.models_mlp import make_mlp
from src.evaluate import evaluate, summarize, threshold_table
from src.data import make_preprocessor
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from scipy.stats import loguniform
from sklearn.model_selection import ParameterSampler
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, recall_score, f1_score
import matplotlib.pyplot as plt

import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)


# Part A: Binary task (Machine failure)

## A1. Setup and data

The 75/25 stratified split (`random_state=42`) is produced by `load_split("binary")`.
In this notebook we load **only the training set**: all tuning is done with
repeated stratified cross-validation (5 folds × 3 repetitions) inside it.
The test set is not loaded here and is touched only once, in the final evaluation.

**Class imbalance.** Only about 3.4% of the rows are failures (~255 of ~7500 in training).
Accuracy is therefore misleading: a model that always predicts "no failure" already
reaches ~96.6%. For this reason we judge models by **precision, recall and F1 of
class 1**, and we track the confusion-matrix counts (FN = missed failures,
FP = false alarms).


In [ ]:
X_train, _, y_train, _ = load_split("binary") #loading just the train split
print(X_train.shape, y_train.mean())

## A2. Early Stopping

Before tuning, we check whether scikit-learn's built-in early stopping is usable.
We compare the same architecture, `(16,)`, 5×3 CV, threshold 0.5, with and without it.

In [ ]:
mlp_es = make_pipeline(
    make_preprocessor(),
    MLPClassifier(hidden_layer_sizes=(16,), early_stopping=True, max_iter=300, random_state=42),
)
results_es, probas_es = evaluate(mlp_es, X_train, y_train)
display(pd.DataFrame([summarize(results_es)]).T)


**Early stopping fails here.** `MLPClassifier` holds out 10% of the training data and
stops when the validation **accuracy** stops improving. With ~3.4% failures, predicting
"no failure" for every row already gives ~96.6% accuracy, so this score plateaus after
a few epochs and training is cut. The result is a model that almost never predicts a failure (low recall).

In [ ]:
results_base, probas_base = evaluate(make_mlp(), X_train, y_train)
display(pd.DataFrame([summarize(results_base)]).T)

**Decision.** We set `early_stopping=False`, so training stops when the training loss stops
improving or at `max_iter`. `max_iter` is therefore a hyperparameter we tune.

## A3. Search space and search strategy

Instead of testing every combination of hyperparameters (grid search), we use
**random search**: we draw a fixed number of configurations (30) at random from the
search space below and evaluate each one with the same repeated stratified 5-fold CV
(3 repeats, seed 42), so all configurations see exactly the same folds.

A grid search must evaluate every combination. There are **525 configurations**, each trained 15 times (5 folds × 3 repeats): 7,875 trainings, against 450 for 30 random configurations.

Random search is also usually as good as or better than grid search for the same
budget. A grid repeats the same few values of each one many times, while random search
tests a new value of every hyperparameter in every trial.

Many configurations will differ by less than the standard deviation across folds , so we do not expect a clear single winner; ties are broken in favour of fewer false alarms, and then of the simpler model.


In [ ]:
#options available for random search
SPACE = {
    "hidden_layer_sizes": [(16,), (32,), (64,), (128,), (32, 16), (64, 32), (64, 32, 16)],
    "alpha": loguniform(1e-5, 1e-1),
    "learning_rate_init": loguniform(1e-4, 1e-2),
    "max_iter": [300, 600, 1000],
}

# Create 30 diferent combinations of SPACE
configs = list(ParameterSampler(SPACE, n_iter=30, random_state=42))
display(pd.DataFrame(configs))

## A4. Evaluating the sampled configurations

**Check on a single configuration.** Before running the whole search, we evaluate
the first sampled configuration with `evaluate` (5×3 repeated stratified CV on the
training set only) and summarize it at eight decision thresholds.
Thresholds are deliberately concentrated below 0.5 (0.1 to 0.5), because with
~3.4% failures the default cut-off favours precision and misses many failures; our
priority is recall.
The predicted probabilities of every fold are stored, so changing the threshold needs
no retraining.

In [ ]:
THRESHOLDS = [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5]

#validate with config 1/30
cfg = configs[0]
t0 = time.time()
res, probas = evaluate(make_mlp(**cfg), X_train, y_train)
print(f"{time.time() - t0:.0f} s")

tt = threshold_table(probas, THRESHOLDS) # summarize the k-fold results and try with diferent thresholds  
display(tt)

**Evaluation of one configuration.** `run_config(name, config, thresholds)` builds the
MLP with `make_mlp(**config)`, runs the CV and returns a table with one row per
threshold (mean and std over the 15 folds of accuracy, precision, recall and F1 of
class 1, plus FN and FP counts), labelled with the configuration name, its
hyperparameters and the training time, and the per-fold probabilities, so we can try with differents thresholds.


In [ ]:
#THRESHOLDS = [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5]

def run_config(name, config, thresholds):
    model = make_mlp(**config) #create the model 

    #timestamp
    t0 = time.time()
    results, probas = evaluate(model, X_train, y_train)
    elapsed = time.time() - t0

    #create the threshold tables (summarized results in diferent thresholds)
    tt = threshold_table(probas, thresholds) 

    #create the column name and seconds for the model
    tt.insert(0, "config", name)
    for k, v in config.items():
        tt[k] = str(v)
    tt["seconds"] = round(elapsed)

    return  tt, probas

#tt, probas = run_config("config_0", cfg, THRESHOLDS)


**Naming and reference model.** Each sampled configuration gets a name
(`cfg_01` … `cfg_30`). We prepend a `baseline` (the `make_mlp` defaults: `(16,)`,
alpha 1e-4, lr 1e-3, 300 epochs) so we can check whether the search really improves
on it: 31 configurations in total.


In [ ]:
# create a name for each config in the list "configs" -> cfg_01 ... cfg_n
random_named = [(f"cfg_{i:02d}", cfg) for i, cfg in enumerate(configs, start=1)]

# standart model of the function make_mlp
baseline = {"hidden_layer_sizes": (16,), "alpha": 1e-4,
            "learning_rate_init": 1e-3, "max_iter": 300}
# append baseline to the other configs
named_configs = [("baseline", baseline)] + random_named
print(named_configs)

**Full run.** Every configuration is evaluated with the same folds (seed 42), at all
eight thresholds, giving 31 × 8 = 248 rows in `summary`. No test data is used.
Results are deterministic (fixed seeds), so rerunning the notebook reproduces them.

In [ ]:
all_tables, all_probas, failed = [], {}, []

for i, (name, cfg) in enumerate(named_configs, start=1):
    #do the evaluate for the config in each threshold
    tt, probas = run_config(name, cfg, THRESHOLDS)
    #add the results to each table
    all_tables.append(tt)
    all_probas[name] = probas
    #print for checking
    print(f"{i}/{len(named_configs)}  {name}  {tt['seconds'].iloc[0]} s")

summary = pd.concat(all_tables, ignore_index=True)

In [ ]:
num_cols = ["alpha", "learning_rate_init", "max_iter"]
summary[num_cols] = summary[num_cols].astype(float)

## A5. Model comparison and selection

We rank the 31 configurations using only cross-validation on the training set.
The selection rule is: among the (configuration, threshold) pairs whose mean
validation recall is at least 0.75, pick the one with the highest F1; the test set
plays no part in it.

**Highest F1.** We first look at the ten (configuration, threshold) pairs with the
highest mean F1 (seed 42). F1 lies between 0.740 and 0.752, with a standard deviation
across folds of 0.03 to 0.06, so these ten are statistically indistinguishable. They
come from only a few configurations (`cfg_03`, `cfg_17`, `cfg_24`, `cfg_27`), all with
two or three hidden layers, and none from the baseline. However, ranking by F1 alone
favours high thresholds (0.35 to 0.50), where recall is only 0.68 to 0.74 and 14 to 16
failures per fold are missed. Since a missed failure is worse than a false alarm, F1
alone is not a sufficient criterion.

In [ ]:
top = summary.sort_values("f1_mean", ascending=False).head(10)
display(top[["config", "threshold", "precision_mean", "recall_mean", "f1_mean", "f1_std", "fn_mean", "fp_mean", "hidden_layer_sizes"]])


**Recall floor.** The brief does not give the relative cost of a missed failure (FN)
and a false alarm (FP), so we assume that a missed failure is more costly (an
undetected failure means unplanned downtime, while a false alarm means one extra
inspection). We therefore require a minimum mean recall and, among the thresholds that
meet it, choose the highest F1. The table above shows how the best option changes with
the floor:

Going from 0.70 to 0.75 removes about 2.7 missed failures at the price of about 7 more
false alarms; going from 0.75 to 0.80 removes about 2.5 more but costs about 14 extra
false alarms. The price of each avoided FN roughly doubles beyond 0.75, so we fix the
floor at **0.75**. This value is a judgement based on the assumed cost asymmetry, not
a result derived from the data.

In [ ]:
for r_min in [0.65, 0.70, 0.75, 0.80]:
    ok = summary[summary["recall_mean"] >= r_min]
    best = ok.loc[ok.groupby("config")["f1_mean"].idxmax()]
    print(f"R_min={r_min}: {len(best)} configs passam")
    display(best.sort_values("f1_mean", ascending=False).head(5)
            [["config", "threshold", "precision_mean", "recall_mean", "f1_mean", "fn_mean", "fp_mean"]])


**Re-evaluation with more seeds.** The five configurations that pass the 0.75 floor
(`cfg_17`, `cfg_04`, `cfg_03`, `cfg_13`, `cfg_24`) and the baseline were re-evaluated
with three additional CV seeds (7, 123 and 2026), on top of the seed 42 used in the
search: 4 × (5 folds × 3 repeats) = 60 folds per configuration. The tables report the
mean over the four seeds and the standard deviation between seeds. This makes the
numbers less dependent on one particular fold partition (and slightly less
optimistic, since the best pairs were selected on seed 42 alone).

In [ ]:
cfg_by_name = dict(named_configs)

# The most promissor configs 
names = ["baseline", "cfg_04", "cfg_03", "cfg_13", "cfg_24", "cfg_17"]
cols = ["config", "seed", "threshold", "precision_mean", "recall_mean",
        "f1_mean", "f1_std", "fn_mean", "fp_mean"]

finalists = []

# seed 42: come from summary
s42 = summary[summary["config"].isin(names)].copy()
s42.insert(1, "seed", 42)
# insert the seed 42 information
finalists.append(s42[cols])

# new seeds
for seed in [7, 123, 2026]:
    for name in names:
        res, probas = evaluate(make_mlp(**cfg_by_name[name]), X_train, y_train, random_state=seed)
        tt = threshold_table(probas, THRESHOLDS)
        tt.insert(0, "config", name)
        tt.insert(1, "seed", seed)
        finalists.append(tt[cols])

all_seeds = pd.concat(finalists, ignore_index=True)
display(all_seeds)


**Selected model: `cfg_17`, threshold 0.25.** Architecture `(64, 32, 16)`, alpha ≈ 0.051,
learning rate ≈ 0.007, 600 epochs. After averaging over the four seeds, it has the
highest F1 among the configurations that meet the recall floor
(F1 = 0.726, recall = 0.753, precision = 0.713).

Compared with the runner-up (`cfg_03` @ 0.25: F1 = 0.712, recall = 0.763), it produces
about 3 fewer false alarms per fold (16.3 against 19.5) in exchange for about 0.5
more missed failures (12.6 against 12.0). With our assumed cost asymmetry this trade is
favourable as long as a missed failure costs less than about 6 false alarms.


In [ ]:
agg = (all_seeds
       .groupby(["config", "threshold"])
       .agg(recall=("recall_mean", "mean"), recall_sd=("recall_mean", "std"),
            f1=("f1_mean", "mean"),         f1_sd=("f1_mean", "std"),
            precision=("precision_mean", "mean"),
            fn=("fn_mean", "mean"), fp=("fp_mean", "mean"))
       .reset_index())
ok = agg[agg["recall"] >= 0.75]
best = ok.loc[ok.groupby("config")["f1"].idxmax()].sort_values("f1", ascending=False)
display(best)

## A6. Learning curves

To inspect the training dynamics of the selected configuration (`cfg_17`) we train it
epoch by epoch with `partial_fit` on 80% of the training set and track the log loss on
that part and on the remaining 20% (one stratified split, so the curves are
illustrative; the selection itself was done with cross-validation). We also track
recall and F1 on the held-out 20% at the chosen threshold (0.25).

The curves cover the **65 epochs that the model actually trains**: `max_iter`
is only a cap, and `MLPClassifier` stops when the training loss does not improve by
more than `tol` for 10 consecutive epochs. For this configuration, training stops
between about 55 and 90 epochs in every cross-validation fold, so changing `max_iter`
from 200 to 600 gives exactly the same results.

**Reading.** Both losses fall quickly in the first epochs and the validation loss
reaches its lowest level after about 30 to 50 epochs, close to where the stopping
rule ends training. Recall and F1 on the validation part rise during the first epochs and then fluctuate.

**Conclusion.** The configuration is neither under-trained nor trained into strong
overfitting. We keep the choice made by cross-validation unchanged.


In [ ]:
X_tr, X_va, y_tr, y_va = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)

pipe = make_mlp(**cfg_by_name["cfg_17"])
pre, mlp = pipe[:-1], pipe[-1]

Xt = pre.fit_transform(X_tr)
Xv = pre.transform(X_va)
print(Xt.shape, Xv.shape, y_tr.mean(), y_va.mean())

# epochs the model really trains before the stopping rule ends the training
ref = make_mlp(**cfg_by_name["cfg_17"]).fit(X_tr, y_tr)
N_EPOCHS = ref[-1].n_iter_
print("Epochs actually trained:", N_EPOCHS)

THR = 0.25
hist = []

for epoch in range(1, N_EPOCHS + 1):
    mlp.partial_fit(Xt, y_tr, classes=[0, 1])      # uma época

    p_tr = mlp.predict_proba(Xt)
    p_va = mlp.predict_proba(Xv)
    pred_va = (p_va[:, 1] >= THR).astype(int)

    hist.append({
        "epoch": epoch,
        "loss_train": log_loss(y_tr, p_tr, labels=[0, 1]),
        "loss_val":   log_loss(y_va, p_va, labels=[0, 1]),
        "recall_val": recall_score(y_va, pred_va, zero_division=0),
        "f1_val":     f1_score(y_va, pred_va, zero_division=0),
    })

hist = pd.DataFrame(hist)
display(hist)


In [ ]:
print("Final epoch:", N_EPOCHS,
      "| loss_train =", round(hist["loss_train"].iloc[-1], 4),
      "| loss_val =", round(hist["loss_val"].iloc[-1], 4),
      "| recall_val =", round(hist["recall_val"].iloc[-1], 3),
      "| f1_val =", round(hist["f1_val"].iloc[-1], 3))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))

ax[0].plot(hist["epoch"], hist["loss_train"], label="train")
ax[0].plot(hist["epoch"], hist["loss_val"], label="validation")
ax[0].set(xlabel="epoch", ylabel="log loss", title="Loss")
ax[0].legend()

ax[1].plot(hist["epoch"], hist["recall_val"], label="recall")
ax[1].plot(hist["epoch"], hist["f1_val"], label="F1")
ax[1].set(xlabel="epoch", ylabel="score (validation)", title=f"Recall and F1 (threshold {THR})")
ax[1].legend()

plt.tight_layout()
plt.show()


## A7. Chosen MLP

Proposed configuration to freeze for the MLP (binary task), chosen only with
cross-validation on the training set:

- **`cfg_17`**: hidden layers `(64, 32, 16)`, alpha ≈ 0.051, learning rate ≈ 0.007,
  `max_iter` = 600 (an upper cap; training stops earlier by the loss-based rule),
  `early_stopping=False`.
- **Decision threshold: 0.25**, selected on validation.
- **Selection rule:** highest F1 among the (configuration, threshold) pairs with mean
  validation recall ≥ 0.75, averaged over four CV seeds.

The table compares it with the runner-up (`cfg_03`) and the baseline.
Compared with the baseline, `cfg_17` misses the same number of failures but raises
about half as many false alarms. The test set has not been used for any choice: the
final training on the full training set and the single test evaluation happen after
the model-selection meeting.


In [ ]:
n_pos = y_train.sum() / 5
n_neg = len(y_train) / 5 - n_pos
cand = best[best["config"].isin(["cfg_17", "cfg_03", "baseline"])].copy()
cand = cand.set_index("config").loc[["cfg_17", "cfg_03", "baseline"]]
cand = cand.assign(tp=n_pos - cand["fn"], tn=n_neg - cand["fp"])
display(cand[["threshold", "precision", "recall", "recall_sd", "f1", "f1_sd", "fn", "fp", "tn", "tp"]].round(3))